
# Upload Data Layers to Hugging Face

Upload raw data (`01_data_layer/raw/`) and engineered feature files (`02_feature_layer/training/outputs/`) to the `PropertyLens/Resealeflats` dataset repository with version control.

## What gets uploaded

### 01_data_layer/raw/
- `ResaleFlatPrices/` — HDB resale transaction CSVs
- `SoldandRentedHDBPropertiesandFacilities/` — HDB property stats (backup files excluded)
- `google_geo/` — Geocoded POI, MRT, school, mall, park distance files
- `schools/` — MOE school data (backup files excluded)
- `raw_collection_metadata_20260412.json`
- `VERSION_MANIFEST.json`

### 02_feature_layer/training/outputs/
- `hdb_feature_table_*.csv`, `hdb_feature_train_*.csv`, `hdb_feature_test_*.csv`
- `feature_metadata_*.json`
- `VERSION_MANIFEST.json`

## Version Control
Each upload compares the local `VERSION_MANIFEST.json` against the copy in HF.  
If the local version is newer (or no remote manifest exists), files are uploaded and the manifest is refreshed.  
If versions match, the upload is skipped.

## Auth
Set `HF_TOKEN` (write-enabled) in `.env` before running:
```
HF_TOKEN=hf_...your_token...
```


In [1]:
%pip install -q python-dotenv huggingface-hub

Note: you may need to restart the kernel to use updated packages.


## Setup and Configuration

In [2]:
import os
import json
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import EntryNotFoundError

# ── Resolve repo root (works from repo root OR any sublayer folder) ──────────
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "hf_data").exists() or (cwd / "01_data_layer").exists() else cwd.parent

# ── Load .env ────────────────────────────────────────────────────────────────
load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(cwd / ".env", override=False)

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Missing HF_TOKEN. Put it in .env or export it in your shell."

# ── Hugging Face repo config ─────────────────────────────────────────────────
HF_REPO_ID   = os.environ.get("HF_REPO_ID",   "PropertyLens/Resealeflats")
HF_REPO_TYPE = os.environ.get("HF_REPO_TYPE", "dataset")
HF_PRIVATE   = os.environ.get("HF_PRIVATE", "true").lower() in ("1", "true", "yes")

# ── Local source directories ──────────────────────────────────────────────────
DATA_RAW_DIR     = REPO_ROOT / "01_data_layer" / "raw"
FEATURE_OUT_DIR  = REPO_ROOT / "02_feature_layer" / "training" / "outputs"

assert DATA_RAW_DIR.exists(),    f"Missing: {DATA_RAW_DIR}"
assert FEATURE_OUT_DIR.exists(), f"Missing: {FEATURE_OUT_DIR}"

print(f"Repo root      : {REPO_ROOT}")
print(f"01 data layer  : {DATA_RAW_DIR}")
print(f"02 feature layer: {FEATURE_OUT_DIR}")
print(f"HF repo        : {HF_REPO_ID}  (type={HF_REPO_TYPE}, private={HF_PRIVATE})")

Repo root      : /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens
01 data layer  : /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw
02 feature layer: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/02_feature_layer/training/outputs
HF repo        : PropertyLens/Resealeflats  (type=dataset, private=True)


/opt/homebrew/anaconda3/envs/nus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Scan Local Files

Preview every file that will be uploaded, grouped by layer.

In [3]:
def _is_backup(path: Path) -> bool:
    """Return True for files that contain '_backup_' in their name (not for upload)."""
    return "_backup_" in path.name


def collect_data_layer_files(base: Path) -> list[tuple[Path, str]]:
    """
    Walk 01_data_layer/raw/ and return (local_path, path_in_repo) pairs.
    Excludes backup files and the logs/ directory.
    """
    entries = []
    for p in sorted(base.rglob("*")):
        if not p.is_file():
            continue
        if _is_backup(p):
            continue
        # Skip the empty logs directory
        if "logs" in p.parts:
            continue
        rel = p.relative_to(base.parent.parent)   # relative to REPO_ROOT
        entries.append((p, str(rel)))
    return entries


def collect_feature_layer_files(base: Path) -> list[tuple[Path, str]]:
    """
    Walk 02_feature_layer/training/outputs/ and return (local_path, path_in_repo) pairs.
    """
    entries = []
    for p in sorted(base.rglob("*")):
        if not p.is_file():
            continue
        rel = p.relative_to(base.parent.parent.parent)  # relative to REPO_ROOT
        entries.append((p, str(rel)))
    return entries


data_files    = collect_data_layer_files(DATA_RAW_DIR)
feature_files = collect_feature_layer_files(FEATURE_OUT_DIR)

def _show(label: str, files: list[tuple[Path, str]]) -> None:
    total_mb = sum(p.stat().st_size for p, _ in files) / 1e6
    print(f"\n{'─' * 70}")
    print(f"  {label}  ({len(files)} files, {total_mb:.1f} MB total)")
    print(f"{'─' * 70}")
    for local, repo_path in files:
        size_mb = local.stat().st_size / 1e6
        print(f"  {repo_path:<70s}  {size_mb:>7.2f} MB")

_show("01_data_layer/raw", data_files)
_show("02_feature_layer/training/outputs", feature_files)

all_files = data_files + feature_files
grand_total_mb = sum(p.stat().st_size for p, _ in all_files) / 1e6
print(f"\n{'=' * 70}")
print(f"  Grand total: {len(all_files)} files  /  {grand_total_mb:.1f} MB")
print(f"{'=' * 70}")


──────────────────────────────────────────────────────────────────────
  01_data_layer/raw  (22 files, 46.2 MB total)
──────────────────────────────────────────────────────────────────────
  01_data_layer/raw/ResaleFlatPrices/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv     3.70 MB
  01_data_layer/raw/ResaleFlatPrices/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv     5.18 MB
  01_data_layer/raw/ResaleFlatPrices/Resale flat prices based on registration date from Jan-2017 onwards.csv    22.42 MB
  01_data_layer/raw/SoldandRentedHDBPropertiesandFacilities/Number of Sold and Rented HDB Commercial Properties.csv     0.01 MB
  01_data_layer/raw/SoldandRentedHDBPropertiesandFacilities/Number of Sold and Rented HDB Industrial Properties.csv     0.01 MB
  01_data_layer/raw/SoldandRentedHDBPropertiesandFacilities/Number of Sold and Rented HDB Residential Units.csv     0.01 MB
  01_data_layer/raw/SoldandRentedHDBPropertiesandFaci

## Version Control Check

Compare the local `VERSION_MANIFEST.json` for each layer against the copy stored in the HF repo.  
Upload is **skipped** for a layer if the remote version already matches the local version.

In [4]:
import tempfile

api = HfApi(token=HF_TOKEN)

# ── Ensure the HF repo exists ─────────────────────────────────────────────────
repo_url = api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    private=HF_PRIVATE,
    exist_ok=True,
)
print(f"HF repo ready: {repo_url}\n")


def fetch_remote_manifest(repo_path_in_hf: str) -> dict | None:
    """Download and parse a VERSION_MANIFEST.json from the HF repo. Returns None if absent."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            local = hf_hub_download(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                filename=repo_path_in_hf,
                local_dir=tmp,
                token=HF_TOKEN,
                force_download=True,
            )
            return json.loads(Path(local).read_text())
    except EntryNotFoundError:
        return None
    except Exception as e:
        print(f"  Warning: could not fetch remote manifest ({e})")
        return None


# ── Paths of manifests in the repo ───────────────────────────────────────────
DATA_MANIFEST_HF    = "01_data_layer/raw/VERSION_MANIFEST.json"
FEATURE_MANIFEST_HF = "02_feature_layer/training/outputs/VERSION_MANIFEST.json"

local_data_manifest    = json.loads((DATA_RAW_DIR    / "VERSION_MANIFEST.json").read_text())
local_feature_manifest = json.loads((FEATURE_OUT_DIR / "VERSION_MANIFEST.json").read_text())

remote_data_manifest    = fetch_remote_manifest(DATA_MANIFEST_HF)
remote_feature_manifest = fetch_remote_manifest(FEATURE_MANIFEST_HF)


def check_needs_upload(label: str, local_m: dict, remote_m: dict | None) -> bool:
    local_ver  = local_m.get("version", "unknown")
    remote_ver = remote_m.get("version", "none") if remote_m else "none"
    needs = (remote_m is None) or (local_ver != remote_ver)
    status = "UPLOAD NEEDED" if needs else "UP TO DATE — skipping"
    print(f"  {label}")
    print(f"    local version : {local_ver}")
    print(f"    remote version: {remote_ver}")
    print(f"    → {status}\n")
    return needs

print("Version comparison:")
print("=" * 70)
upload_data    = check_needs_upload("01_data_layer",          local_data_manifest,    remote_data_manifest)
upload_features = check_needs_upload("02_feature_layer",      local_feature_manifest, remote_feature_manifest)

HF repo ready: https://huggingface.co/datasets/PropertyLens/Resealeflats

Version comparison:
  01_data_layer
    local version : 20260412
    remote version: 20260406
    → UPLOAD NEEDED

  02_feature_layer
    local version : 20260412
    remote version: 20260403
    → UPLOAD NEEDED



## Upload 01_data_layer

Upload every file under `01_data_layer/raw/` (backup files and `logs/` excluded).

In [5]:
data_successful = 0
data_failed = 0

if not upload_data:
    print("01_data_layer is already up to date on HF. Skipping.")
else:
    commit_msg = f"Upload 01_data_layer raw data  [v{local_data_manifest.get('version', 'unknown')}]"
    print(f"{'=' * 70}")
    print(f"Uploading 01_data_layer ({len(data_files)} files)...")
    print(f"Commit message: {commit_msg}\n")

    for idx, (local_path, repo_path) in enumerate(data_files, 1):
        size_mb = local_path.stat().st_size / 1e6
        print(f"[{idx:>3}/{len(data_files)}] {repo_path}  ({size_mb:.2f} MB)")
        try:
            api.upload_file(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                path_or_fileobj=str(local_path),
                path_in_repo=repo_path,
                commit_message=commit_msg,
                token=HF_TOKEN,
            )
            print(f"          ✓")
            data_successful += 1
        except Exception as e:
            print(f"          ✗  Error: {e}")
            data_failed += 1

    print(f"\n01_data_layer upload complete  ✓ {data_successful}  ✗ {data_failed}")

Uploading 01_data_layer (22 files)...
Commit message: Upload 01_data_layer raw data  [v20260412]

[  1/22] 01_data_layer/raw/ResaleFlatPrices/Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv  (3.70 MB)


No files have been modified since last commit. Skipping to prevent empty commit.


          ✓
[  2/22] 01_data_layer/raw/ResaleFlatPrices/Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv  (5.18 MB)


No files have been modified since last commit. Skipping to prevent empty commit.


          ✓
[  3/22] 01_data_layer/raw/ResaleFlatPrices/Resale flat prices based on registration date from Jan-2017 onwards.csv  (22.42 MB)


...

          ✓
[  8/22] 01_data_layer/raw/VERSION_MANIFEST.json  (0.00 MB)
          ✓
[  9/22] 01_data_layer/raw/google_geo/hdb_geo_accessibility_noise_features_20260412.csv  (3.32 MB)
          ✓
[ 10/22] 01_data_layer/raw/google_geo/moe_school_geocode_20260412.csv  (0.03 MB)
          ✓
[ 11/22] 01_data_layer/raw/google_geo/nea_hawker_centres_20260412.csv  (0.02 MB)
          ✓
[ 12/22] 01_data_layer/raw/google_geo/onemap_geocode_checkpoint.csv  (0.32 MB)


No files have been modified since last commit. Skipping to prevent empty commit.


          ✓
[ 13/22] 01_data_layer/raw/google_geo/onemap_geocode_raw_responses.jsonl  (6.14 MB)


No files have been modified since last commit. Skipping to prevent empty commit.


          ✓
[ 14/22] 01_data_layer/raw/google_geo/onemap_hdb_geocode_with_highway_dist_20260412.csv  (2.68 MB)
          ✓
[ 15/22] 01_data_layer/raw/google_geo/onemap_mall_nodes_20260412.csv  (0.02 MB)
          ✓
[ 16/22] 01_data_layer/raw/google_geo/onemap_mrt_lrt_nodes_20260412.csv  (0.02 MB)
          ✓
[ 17/22] 01_data_layer/raw/google_geo/onemap_parks_playgrounds_20260412.csv  (0.03 MB)
          ✓
[ 18/22] 01_data_layer/raw/google_geo/onemap_transit_nodes_20260412.csv  (0.02 MB)
          ✓
[ 19/22] 01_data_layer/raw/raw_collection_metadata_20260412.json  (0.00 MB)
          ✓
[ 20/22] 01_data_layer/raw/schools/moe_general_information_of_schools_20260412.csv  (0.15 MB)
          ✓
[ 21/22] 01_data_layer/raw/schools/moe_schools_geocode_20260412.csv  (0.15 MB)
          ✓
[ 22/22] 01_data_layer/raw/schools/sgschooling_2015plus_20260412.csv  (1.92 MB)
          ✓

01_data_layer upload complete  ✓ 22  ✗ 0


## Upload 02_feature_layer

Upload every file under `02_feature_layer/training/outputs/`.

In [6]:
feat_successful = 0
feat_failed = 0

if not upload_features:
    print("02_feature_layer is already up to date on HF. Skipping.")
else:
    commit_msg = f"Upload 02_feature_layer outputs  [v{local_feature_manifest.get('version', 'unknown')}]"
    print(f"{'=' * 70}")
    print(f"Uploading 02_feature_layer ({len(feature_files)} files)...")
    print(f"Commit message: {commit_msg}\n")

    for idx, (local_path, repo_path) in enumerate(feature_files, 1):
        size_mb = local_path.stat().st_size / 1e6
        print(f"[{idx:>3}/{len(feature_files)}] {repo_path}  ({size_mb:.2f} MB)")
        try:
            api.upload_file(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                path_or_fileobj=str(local_path),
                path_in_repo=repo_path,
                commit_message=commit_msg,
                token=HF_TOKEN,
            )
            print(f"          ✓")
            feat_successful += 1
        except Exception as e:
            print(f"          ✗  Error: {e}")
            feat_failed += 1

    print(f"\n02_feature_layer upload complete  ✓ {feat_successful}  ✗ {feat_failed}")

Uploading 02_feature_layer (5 files)...
Commit message: Upload 02_feature_layer outputs  [v20260412]

[  1/5] 02_feature_layer/training/outputs/VERSION_MANIFEST.json  (0.00 MB)
          ✓
[  2/5] 02_feature_layer/training/outputs/feature_metadata_20260412.json  (0.00 MB)
          ✓
[  3/5] 02_feature_layer/training/outputs/hdb_feature_table_20260412.csv  (172.03 MB)


Processing Files (1 / 1): 100%|██████████|  172MB /  172MB, 17.2MB/s  
New Data Upload: 100%|██████████|  172MB /  172MB, 17.2MB/s  


          ✓
[  4/5] 02_feature_layer/training/outputs/hdb_feature_test_20260412.csv  (54.22 MB)


Processing Files (1 / 1): 100%|██████████| 54.2MB / 54.2MB, 30.1MB/s  
New Data Upload: 100%|██████████|  203kB /  203kB,  113kB/s  


          ✓
[  5/5] 02_feature_layer/training/outputs/hdb_feature_train_20260412.csv  (117.81 MB)


Processing Files (1 / 1): 100%|██████████|  118MB /  118MB, 77.6kB/s  
New Data Upload: 100%|██████████|  109kB /  109kB, 77.6kB/s  


          ✓

02_feature_layer upload complete  ✓ 5  ✗ 0


## Upload Summary

In [7]:
print("=" * 70)
print("Upload Summary")
print("=" * 70)

if upload_data:
    print(f"  01_data_layer    → ✓ {data_successful:>3} uploaded   ✗ {data_failed:>3} failed")
else:
    print(f"  01_data_layer    → skipped (already up to date)")

if upload_features:
    print(f"  02_feature_layer → ✓ {feat_successful:>3} uploaded   ✗ {feat_failed:>3} failed")
else:
    print(f"  02_feature_layer → skipped (already up to date)")

total_ok  = (data_successful if upload_data else 0) + (feat_successful if upload_features else 0)
total_err = (data_failed     if upload_data else 0) + (feat_failed     if upload_features else 0)
print(f"\n  Total: ✓ {total_ok} uploaded   ✗ {total_err} failed")
print(f"\n  Repo: https://huggingface.co/datasets/{HF_REPO_ID}")

Upload Summary
  01_data_layer    → ✓  22 uploaded   ✗   0 failed
  02_feature_layer → ✓   5 uploaded   ✗   0 failed

  Total: ✓ 27 uploaded   ✗ 0 failed

  Repo: https://huggingface.co/datasets/PropertyLens/Resealeflats
